# Implicit Regularization Study: Sven vs Adam / AdamW / SGD

**Hypothesis**: Sven's pseudo-inverse update δθ = −η M⁺ ℓ yields the minimum-norm solution in the overparameterized regime, introducing implicit regularization that improves generalization. The SVD rank *k* acts as an explicit regularization knob.

**Tasks**: random polynomial regression and toy 1D regression, both in the overfit regime (small training sets, large models).

**Run experiments first**:
```bash
python -m experiments.run_experiment --config-name implicit_reg_polynomial
python -m experiments.run_experiment --config-name implicit_reg_toy1d
```

## 0. Setup & Data Loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'sv3'))
from style import set_style
set_style()

RESULTS_DIR = '../experiment_results'
PLOT_DIR = 'plots/implicit_regularization'
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
def load_jsonl(name):
    path = os.path.join(RESULTS_DIR, f'{name}.jsonl')
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


def get_loss_array(row, key):
    """Extract a loss array from the nested losses dict."""
    v = row['losses'].get(key, [])
    return np.array(v) if v else np.array([])


def final_loss(row, key):
    arr = get_loss_array(row, key)
    return float(arr[-1]) if len(arr) > 0 else np.nan


def add_derived_metrics(df):
    df = df.copy()
    df['final_train_loss'] = df.apply(lambda r: final_loss(r, 'train'), axis=1)
    df['final_val_loss']   = df.apply(lambda r: final_loss(r, 'val'),   axis=1)
    df['final_param_norm'] = df.apply(lambda r: final_loss(r, 'param_norm'), axis=1)
    df['gen_gap'] = df['final_val_loss'] - df['final_train_loss']
    df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
    return df


def best_run(df, group_col, metric='final_val_loss'):
    """Return the row with the minimum metric value for each group."""
    idx = df.groupby(group_col)[metric].idxmin()
    return df.loc[idx].reset_index(drop=True)


# Optimizer display names and colors
OPT_COLORS = {
    'SVD':   '#1f77b4',
    'Adam':  '#ff7f0e',
    'AdamW': '#2ca02c',
    'SGD':   '#d62728',
}

print('Helpers loaded.')

In [ ]:
df_poly_raw = load_jsonl('implicit_reg_polynomial')
df_1d_raw   = load_jsonl('implicit_reg_toy1d')

df_poly = add_derived_metrics(df_poly_raw)
df_1d   = add_derived_metrics(df_1d_raw)

print('Polynomial runs:', len(df_poly), '| optimizers:', df_poly['optimizer'].unique())
print('Toy 1D runs:    ', len(df_1d),   '| optimizers:', df_1d['optimizer'].unique())

## 1. Best Configs — Final Validation Loss

Best hyperparameters (lr, k, rtol / weight_decay) selected per optimizer by lowest final val loss, averaged over seeds.

In [ ]:
def plot_best_val_loss(df, title, ax):
    # For AdamW, create a label that includes weight_decay
    df = df.copy()
    df['opt_label'] = df.apply(
        lambda r: f"AdamW wd={r['weight_decay']:.0e}" if r['optimizer'] == 'AdamW' and r.get('weight_decay', 0) != 0
        else r['optimizer'],
        axis=1
    )
    best = best_run(df, 'opt_label', 'final_val_loss')
    best = best.sort_values('final_val_loss')
    colors = [OPT_COLORS.get(r['optimizer'], 'gray') for _, r in best.iterrows()]
    ax.bar(best['opt_label'], best['final_val_loss'], color=colors)
    ax.set_yscale('log')
    ax.set_ylabel('Final val loss')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_best_val_loss(df_poly, 'Polynomial regression', axes[0])
plot_best_val_loss(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/best_val_loss.pdf')
plt.show()

## 2. Train vs Val Loss Curves

Best hyperparameter config per optimizer. Solid = train, dashed = val. Divergence between the two curves indicates overfitting.

In [ ]:
def plot_train_val_curves(df, title, ax, optimizers=('SVD', 'Adam', 'AdamW', 'SGD')):
    for opt in optimizers:
        sub = df[df['optimizer'] == opt]
        if sub.empty:
            continue
        best_idx = sub['final_val_loss'].idxmin()
        row = sub.loc[best_idx]
        color = OPT_COLORS.get(opt, 'gray')
        label = opt
        if opt == 'SVD':
            label = f"Sven k={int(row['k'])}"
        elif opt == 'AdamW':
            wd = row.get('weight_decay', 0)
            label = f"AdamW wd={wd:.0e}"

        train_arr = get_loss_array(row, 'train')
        val_arr   = get_loss_array(row, 'val')[1:]  # val has t=0 prepended; align with epochs
        epochs = np.arange(1, len(train_arr) + 1)

        ax.plot(epochs, train_arr, color=color, linestyle='-',  label=f'{label} train')
        ax.plot(epochs, val_arr[:len(epochs)],   color=color, linestyle='--', label=f'{label} val')

    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend(ncol=2, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_train_val_curves(df_poly, 'Polynomial regression', axes[0])
plot_train_val_curves(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/train_val_curves.pdf')
plt.show()

## 3. Generalization Gap Over Epochs

gap(t) = val_loss(t) − train_loss(t).  
Mean ± std across seeds for best hyperparameters per optimizer.  
A smaller, flatter gap indicates better implicit regularization.

In [ ]:
def compute_gap_curves(df, opt):
    """Return mean and std of (val - train) gap curves across seeds for best hparams."""
    sub = df[df['optimizer'] == opt]
    if sub.empty:
        return None, None, None
    # Find best hparams by mean final val loss across seeds
    if opt == 'SVD':
        hparam_cols = ['k', 'lr', 'rtol']
    else:
        hparam_cols = ['lr', 'weight_decay']
    hparam_cols = [c for c in hparam_cols if c in sub.columns]
    grouped = sub.groupby(hparam_cols)['final_val_loss'].mean()
    best_hparams = grouped.idxmin()
    if not isinstance(best_hparams, tuple):
        best_hparams = (best_hparams,)
    mask = pd.Series([True] * len(sub), index=sub.index)
    for col, val in zip(hparam_cols, best_hparams):
        mask &= (sub[col] == val)
    best_rows = sub[mask]

    gap_curves = []
    for _, row in best_rows.iterrows():
        train_arr = get_loss_array(row, 'train')
        val_arr   = get_loss_array(row, 'val')[1:len(train_arr)+1]
        if len(val_arr) == len(train_arr):
            gap_curves.append(val_arr - train_arr)
    if not gap_curves:
        return None, None, None
    gap_arr = np.array(gap_curves)
    return np.arange(1, gap_arr.shape[1]+1), gap_arr.mean(0), gap_arr.std(0)


def plot_gen_gap(df, title, ax, optimizers=('SVD', 'Adam', 'AdamW', 'SGD')):
    for opt in optimizers:
        epochs, mean, std = compute_gap_curves(df, opt)
        if epochs is None:
            continue
        color = OPT_COLORS.get(opt, 'gray')
        ax.plot(epochs, mean, color=color, label=opt)
        ax.fill_between(epochs, mean-std, mean+std, color=color, alpha=0.15)
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val − Train loss')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_gen_gap(df_poly, 'Polynomial regression', axes[0])
plot_gen_gap(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/generalization_gap.pdf')
plt.show()

## 4. Parameter Norm Trajectories

‖θ(t)‖₂ for the best config per optimizer.  
Theory predicts Sven maintains a smaller parameter norm due to minimum-norm updates.

In [ ]:
def plot_param_norm(df, title, ax, optimizers=('SVD', 'Adam', 'AdamW', 'SGD')):
    for opt in optimizers:
        sub = df[df['optimizer'] == opt]
        if sub.empty:
            continue
        best_idx = sub['final_val_loss'].idxmin()
        row = sub.loc[best_idx]
        pnorm = get_loss_array(row, 'param_norm')
        if len(pnorm) == 0:
            continue
        color = OPT_COLORS.get(opt, 'gray')
        label = opt
        if opt == 'SVD':
            label = f"Sven k={int(row['k'])}"
        ax.plot(np.arange(1, len(pnorm)+1), pnorm, color=color, label=label)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('‖θ‖₂')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_param_norm(df_poly, 'Polynomial regression', axes[0])
plot_param_norm(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/param_norm.pdf')
plt.show()

## 5. k as Regularization Parameter (Key Plot)

For Sven, lower k means stronger regularization (updates restricted to a lower-dimensional subspace).  
We expect a U-shape: too-low k underfits, optimal k generalizes best, too-high k starts to overfit like standard SGD.

In [ ]:
def plot_k_vs_metric(df, metric, ylabel, title, ax):
    svd = df[df['optimizer'] == 'SVD'].copy()
    # Best lr/rtol per k (mean over seeds)
    best_by_k = svd.groupby(['k', 'lr', 'rtol'])[metric].mean().reset_index()
    best_by_k = best_by_k.loc[best_by_k.groupby('k')[metric].idxmin()]
    ax.plot(best_by_k['k'], best_by_k[metric], marker='o', color=OPT_COLORS['SVD'], label='Sven')
    ax.set_xlabel('SVD rank k')
    ax.set_ylabel(ylabel)
    ax.set_yscale('log')
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_k_vs_metric(df_poly, 'final_val_loss',   'Final val loss',   'Polynomial — val loss vs k',   axes[0,0])
plot_k_vs_metric(df_1d,   'final_val_loss',   'Final val loss',   'Toy 1D — val loss vs k',       axes[0,1])
plot_k_vs_metric(df_poly, 'gen_gap',          'Val − Train loss', 'Polynomial — gen gap vs k',    axes[1,0])
plot_k_vs_metric(df_1d,   'gen_gap',          'Val − Train loss', 'Toy 1D — gen gap vs k',        axes[1,1])

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/k_vs_generalization.pdf')
plt.show()

In [ ]:
# Also: k vs final parameter norm
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_k_vs_metric(df_poly, 'final_param_norm', '‖θ_final‖₂', 'Polynomial — param norm vs k', axes[0])
plot_k_vs_metric(df_1d,   'final_param_norm', '‖θ_final‖₂', 'Toy 1D — param norm vs k',    axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/k_vs_param_norm.pdf')
plt.show()

## 6. Implicit vs Explicit Regularization

Compare Sven (varying k) vs AdamW (varying weight_decay) on the same axes.  
This reveals whether the two forms of regularization are interchangeable, and what k "corresponds to" in terms of explicit L2 regularization strength.

In [ ]:
def plot_implicit_vs_explicit(df, title, ax):
    # Sven: best val loss per k (best lr/rtol)
    svd = df[df['optimizer'] == 'SVD']
    svd_best = svd.groupby(['k', 'lr', 'rtol'])['final_val_loss'].mean().reset_index()
    svd_best = svd_best.loc[svd_best.groupby('k')['final_val_loss'].idxmin()]
    svd_best = svd_best.sort_values('k')
    # Normalize k to [0,1] as a proxy for regularization strength (1/k)
    k_max = svd_best['k'].max()

    # AdamW: best val loss per weight_decay (best lr)
    adamw = df[(df['optimizer'] == 'AdamW')]
    adamw_best = adamw.groupby(['weight_decay', 'lr'])['final_val_loss'].mean().reset_index()
    adamw_best = adamw_best.loc[adamw_best.groupby('weight_decay')['final_val_loss'].idxmin()]
    adamw_best = adamw_best.sort_values('weight_decay')

    # Twin x-axes: bottom = k (Sven), top = weight_decay (AdamW)
    ax.plot(svd_best['k'], svd_best['final_val_loss'],
            marker='o', color=OPT_COLORS['SVD'], label='Sven (vary k)')
    ax2 = ax.twiny()
    ax2.plot(adamw_best['weight_decay'], adamw_best['final_val_loss'],
             marker='s', color=OPT_COLORS['AdamW'], label='AdamW (vary wd)', linestyle='--')
    ax2.set_xscale('log')
    ax2.set_xlabel('AdamW weight_decay →', color=OPT_COLORS['AdamW'])

    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.invert_xaxis()  # higher k = less regularization
    ax.set_xlabel('← Sven rank k (lower = more regularization)', color=OPT_COLORS['SVD'])
    ax.set_ylabel('Final val loss')
    ax.set_yscale('log')
    ax.set_title(title)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_implicit_vs_explicit(df_poly, 'Polynomial regression', axes[0])
plot_implicit_vs_explicit(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/implicit_vs_explicit_reg.pdf')
plt.show()

## 7. Wall-Time Matched Comparisons

At equal wall-clock time, does Sven's implicit regularization come for free or at a compute cost?
Plot val_loss vs cumulative training time.

In [ ]:
def plot_walltime(df, title, ax, optimizers=('SVD', 'Adam', 'AdamW', 'SGD')):
    for opt in optimizers:
        sub = df[df['optimizer'] == opt]
        if sub.empty:
            continue
        best_idx = sub['final_val_loss'].idxmin()
        row = sub.loc[best_idx]
        epoch_times = get_loss_array(row, 'epoch_times')
        val_arr     = get_loss_array(row, 'val')[1:]  # drop t=0
        if len(epoch_times) == 0 or len(val_arr) == 0:
            continue
        cumtime = np.cumsum(epoch_times[:len(val_arr)])
        color = OPT_COLORS.get(opt, 'gray')
        label = opt
        if opt == 'SVD':
            label = f"Sven k={int(row['k'])}"
        ax.plot(cumtime, val_arr[:len(cumtime)], color=color, label=label)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Cumulative time (s)')
    ax.set_ylabel('Val loss')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_walltime(df_poly, 'Polynomial regression', axes[0])
plot_walltime(df_1d,   'Toy 1D regression',     axes[1])
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/walltime_val_loss.pdf')
plt.show()

## 8. Summary Statistics

In [ ]:
def summary_table(df, dataset_name):
    print(f'\n=== {dataset_name} ===')
    opts = df['optimizer'].unique()
    rows = []
    for opt in opts:
        sub = df[df['optimizer'] == opt]
        best_idx = sub['final_val_loss'].idxmin()
        row = sub.loc[best_idx]
        rows.append({
            'Optimizer': opt,
            'Best val loss': f"{row['final_val_loss']:.4e}",
            'Train loss':    f"{row['final_train_loss']:.4e}",
            'Gen gap':       f"{row['gen_gap']:.4e}",
            '‖θ‖₂':         f"{row['final_param_norm']:.3f}" if not np.isnan(row['final_param_norm']) else 'N/A',
            'Time (s)':      f"{row['total_time']:.1f}",
        })
    print(pd.DataFrame(rows).to_string(index=False))

summary_table(df_poly, 'Polynomial regression')
summary_table(df_1d,   'Toy 1D regression')